### First I will check if data is present in datasets.

In [1]:
import pandas as pd

In [2]:
# Define HEART metrics for Stack Overflow
heart_metrics = {
    'Happiness': 'Measure of user satisfaction and happiness with the Questions & Answers product.',
    'Engagement': 'Measure of user interaction and activity on the platform.',
    'Adoption': 'Measure of user acquisition and growth of Stack Overflow user base.',
    'Retention': 'Measure of user retention and continued usage of the platform over time.',
    'Task Success': 'Measure of user ability to accomplish their goals and tasks effectively on Stack Overflow.'
}

In [3]:
file_path = 'datasets/Tags.xml'

In [4]:
tags_df = pd.read_xml(file_path)
tags_df.head()

,Id,TagName,Count,ExcerptPostId,WikiPostId
0,1,.net,337923,3624959.0,3607476.0
1,2,html,1187348,3673183.0,3673182.0
2,3,javascript,2528894,3624960.0,3607052.0
3,4,css,804268,3644670.0,3644669.0
4,5,php,1464496,3624936.0,3607050.0


In [5]:
# users_df = pd.read_xml('datasets/Users.xml')
# users_df.head()

Since there is a memory error with loading 2nd smalest xml file, I need to use another approach. One of them is:

Option 1: Stream XML Instead of Loading It All at Once
Instead of reading the entire file at once, you can iterate over chunks using iterparse. This is the best approach if you just want to process parts of the data without overwhelming your RAM.

Example: Streaming Users.xml

In [6]:
# import xml.etree.ElementTree as ET

# file_path = "datasets/Users.xml"

# # Create an empty list to store data
# data = []

# # Stream the XML file
# for event, elem in ET.iterparse(file_path, events=("start",)):
#     if elem.tag == "row":  # Adjust tag based on XML structure
#         data.append(elem.attrib)  # Extract attributes

#         if len(data) % 100000 == 0:  # Print progress every 100K rows
#             print(f"Processed {len(data)} rows...")

# # Convert to DataFrame
# df = pd.DataFrame(data)
# print(df.head())


But this is taking very long. So I will try the 2nd option:

Option 2: Convert to a SQLite Database for Querying
If your goal is to filter or analyze large files efficiently, converting them into a lightweight SQLite database is a great idea.

In [7]:
import sqlite3
import xml.etree.ElementTree as ET

file_path = "datasets/Users.xml"
conn = sqlite3.connect("stackoverflow.db")  # Create a SQLite database
cursor = conn.cursor()

# Create table
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    Id INTEGER PRIMARY KEY,
    Reputation INTEGER,
    CreationDate TEXT,
    DisplayName TEXT
);
""")

# Insert data in chunks
for event, elem in ET.iterparse(file_path, events=("start",)):
    if elem.tag == "row":
        cursor.execute("INSERT INTO users (Id, Reputation, CreationDate, DisplayName) VALUES (?, ?, ?, ?)", 
                       (elem.attrib.get("Id"), 
                        elem.attrib.get("Reputation"), 
                        elem.attrib.get("CreationDate"), 
                        elem.attrib.get("DisplayName")))
        if int(elem.attrib.get("Id", 0)) % 100000 == 0:
            print(f"Inserted {elem.attrib.get('Id')} rows...")
conn.commit()
conn.close()


Inserted 600000 rows...
Inserted 800000 rows...
Inserted 900000 rows...
Inserted 1100000 rows...
Inserted 1000000 rows...
Inserted 1200000 rows...
Inserted 1400000 rows...
Inserted 1600000 rows...
Inserted 1800000 rows...
Inserted 1700000 rows...
Inserted 2000000 rows...
Inserted 1900000 rows...
Inserted 2200000 rows...
Inserted 2100000 rows...
Inserted 2400000 rows...
Inserted 2500000 rows...
Inserted 2600000 rows...
Inserted 2300000 rows...
Inserted 2700000 rows...
Inserted 2800000 rows...
Inserted 3000000 rows...
Inserted 3100000 rows...
Inserted 2900000 rows...
Inserted 3300000 rows...
Inserted 3400000 rows...
Inserted 3200000 rows...
Inserted 3500000 rows...
Inserted 3600000 rows...
Inserted 3800000 rows...
Inserted 3700000 rows...
Inserted 3900000 rows...
Inserted 4000000 rows...
Inserted 4200000 rows...
Inserted 4100000 rows...
Inserted 4400000 rows...
Inserted 4300000 rows...
Inserted 4600000 rows...
Inserted 4700000 rows...
Inserted 4800000 rows...
Inserted 4500000 rows...
Ins

In [9]:
conn = sqlite3.connect("stackoverflow.db")
df = pd.read_sql("SELECT * FROM users LIMIT 10", conn)
conn.close()
df.head()

,Id,Reputation,CreationDate,DisplayName
0,-1017,1,2023-09-15T20:10:32.247,Mobile Development
1,-1016,1,2023-06-28T20:22:59.967,PHP
2,-1015,1,2023-06-28T18:50:43.493,NLP
3,-1014,1,2023-02-17T19:52:56.213,R Language
4,-1013,1,2023-02-17T19:25:19.953,CI/CD


In [10]:
conn = sqlite3.connect("stackoverflow.db")
df = pd.read_sql("SELECT * FROM users LIMIT 10", conn)

In [11]:
df

,Id,Reputation,CreationDate,DisplayName
0,-1017,1,2023-09-15T20:10:32.247,Mobile Development
1,-1016,1,2023-06-28T20:22:59.967,PHP
2,-1015,1,2023-06-28T18:50:43.493,NLP
3,-1014,1,2023-02-17T19:52:56.213,R Language
4,-1013,1,2023-02-17T19:25:19.953,CI/CD
5,-1012,1,2023-02-15T23:24:13.363,Audio
6,-1011,1,2022-11-09T20:49:03.637,AWS
7,-1010,1,2022-10-25T19:18:31.537,Microsoft Azure
8,-1009,1,2022-05-17T15:06:54.890,WSO2
9,-1008,1,2022-01-31T19:45:27.477,Twilio
